## Setup PyTorch

In [2]:
import torch

In [3]:
# // Inputs
x = torch.tensor(7.6)  # Input feature
y = torch.tensor(0.0)  # True label (binary)
w = torch.tensor(1.0)  # Weight
b = torch.tensor(0.0)  # Bias

## Manual Implementation

In [4]:
# // Binary Cross-Entropy Loss for scalar
def binary_cross_entropy_loss(prediction, target):
    epsilon = 1e-8  # To prevent log(0)
    prediction = torch.clamp(prediction, epsilon, 1 - epsilon)
    return -(target * torch.log(prediction) + (1 - target) * torch.log(1 - prediction))

In [5]:
# // Forward pass
z = w * x + b  # Weighted sum (linear part)
y_pred = torch.sigmoid(z)  # Predicted probability
# Compute binary cross-entropy loss
loss = binary_cross_entropy_loss(y_pred, y)
loss

tensor(7.6005)

In [6]:
# // Derivatives:
# 1. dL/d(y_pred): Loss with respect to the prediction (y_pred)
dloss_dy_pred = (y_pred - y)/(y_pred*(1-y_pred))
# 2. dy_pred/dz: Prediction (y_pred) with respect to z (sigmoid derivative)
dy_pred_dz = y_pred * (1 - y_pred)
# 3. dz/dw and dz/db: z with respect to w and b
dz_dw = x  # dz/dw = x
dz_db = 1  # dz/db = 1 (bias contributes directly to z)
dL_dw = dloss_dy_pred * dy_pred_dz * dz_dw
dL_db = dloss_dy_pred * dy_pred_dz * dz_db

In [7]:
print(f"Manual Gradient of loss w.r.t weight (dw): {dL_dw}")
print(f"Manual Gradient of loss w.r.t bias (db): {dL_db}")

Manual Gradient of loss w.r.t weight (dw): 7.596198558807373
Manual Gradient of loss w.r.t bias (db): 0.9994997978210449


## PyTorch Autograd Implementation

In [8]:
# // Using Autograd
x = torch.tensor(7.6)
y = torch.tensor(0.0)
w = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
z = w*x + b
y_pred = torch.sigmoid(z)
print(x, y, w, b, z)
loss = binary_cross_entropy_loss(y_pred, y)
print(loss, loss.backward())
print(w.grad)
print(b.grad)
# clearing autograd
w.grad.zero_()
b.grad.zero_()

tensor(7.6000) tensor(0.) tensor(1., requires_grad=True) tensor(0., requires_grad=True) tensor(7.6000, grad_fn=<AddBackward0>)
tensor(7.6005, grad_fn=<NegBackward0>) None
tensor(7.5962)
tensor(0.9995)


tensor(0.)

In [9]:
# // Disable Grad Track
with torch.no_grad():
  print(w, "However, the grad is not tracked temporarily")
z = w.detach()*x + b.detach() # grad is not tracked for created entities if founding entities with "requires_grad = True" is passed with ".detach()".
print(z)
print(b)
b.requires_grad_(False)
b

tensor(1., requires_grad=True) However, the grad is not tracked temporarily
tensor(7.6000)
tensor(0., requires_grad=True)


tensor(0.)